# Cognometry — 2-minute `@trust` demo

One decorator. Any LLM. Verified output.

This notebook demonstrates `styxx.trust` catching a hallucination on a live OpenAI call without any configuration — zero-config reference auto-detect, auto-enabled NLI, adaptive threshold, all shipped in styxx 4.0.1.

Manifesto: https://fathom.darkflobi.com/cognometry  
Source: https://github.com/fathom-lab/styxx  
Paper: `papers/cognometry-v0.md`

---

## 1. Install

`[nli]` pulls in the DeBERTa NLI scorer (~184M params). Auto-enabled once installed.

In [ ]:
%pip install -q styxx[nli] openai

## 2. Set your OpenAI key

Colab: **Runtime → Manage secrets → add `OPENAI_API_KEY`**, or paste below.

In [ ]:
import os

# If you added the key via Colab Secrets, this works automatically:
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    # Fallback: paste here if not using Colab Secrets
    # os.environ['OPENAI_API_KEY'] = 'sk-...'
    pass

assert os.environ.get('OPENAI_API_KEY'), 'set OPENAI_API_KEY first'
print('key set')

## 3. Wrap any LLM-calling function with `@trust`

No configuration. `@trust` inspects the function signature, auto-detects `context` as the reference passage, and auto-enables NLI because `styxx[nli]` is installed.

In [ ]:
from styxx import trust
import openai

client = openai.OpenAI()

@trust
def ask(question, *, context):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'user',
             'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely.'},
        ],
        temperature=0.3,
    )
    return r.choices[0].message.content

## 4. Correct answer → passes through unchanged

The model gets the right context and the right question. `@trust` sees a grounded, coherent response. Returns verbatim.

In [ ]:
context = '''Inception is a 2010 science fiction film written and 
directed by Christopher Nolan. It stars Leonardo DiCaprio as a thief 
who steals corporate secrets through dream-sharing technology.'''

ask('Who directed Inception?', context=context)

## 5. Force a hallucination → `@trust` catches it

Same function, but now we stuff a prompt that pushes the model to answer a question whose answer is NOT in the context. `@trust` compares the response's novelty + NLI-contradiction signals against the passage and halts if risk > threshold.

In [ ]:
# Passage doesn't mention the budget. Model will likely make one up.
context = '''Inception is a 2010 science fiction film written and 
directed by Christopher Nolan.'''

# Force a high-confidence fabrication by asking a specific unknown.
ask('What was the exact production budget of Inception, down to the dollar? Reply with just the number.',
    context=context)

If that returned the styxx fallback ("*I'm not confident enough in my answer...*") instead of a fabricated dollar number, the detector worked.

---
## 6. Inspect the verdict — `on_halt='annotate'`

Return `TrustResult` so you can see WHY each call was flagged.

In [ ]:
@trust(on_halt='annotate')
def ask_annotated(question, *, context):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'user',
             'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely.'},
        ],
        temperature=0.3,
    )
    return r.choices[0].message.content

result = ask_annotated(
    'What was the exact production budget of Inception, down to the dollar? Reply with just the number.',
    context=context,
)

print(f'response : {result.response}')
print(f'risk     : {result.verdict.risk:.3f}')
print(f'action   : {result.verdict.action}')
print(f'halted   : {result.halted}')
print(f'attempts : {result.attempts}')
print('\nsignals:')
for s in result.verdict.signals:
    print(f'  {s.name:<22s} {s.value}')

## 7. Why should you trust the detector?

styxx 4.0.1 is cross-validated across **8 public benchmarks**. Per-dataset held-out test AUC (3-seed averaged, n=150/dataset):

| Benchmark | AUC |
|---|---|
| HaluEval-QA | **0.998** |
| TruthfulQA | **0.994** |
| HaluBench-RAGTruth | **0.807** |
| HaluBench-PubMedQA | **0.719** |
| HaluEval-Dialog | 0.676 |
| HaluEval-Summarization | 0.643 |
| HaluBench-FinanceBench | 0.492 — **published failure** |
| HaluBench-DROP | 0.424 — **published failure** |

5/8 above AUC 0.65. Two below-chance results published openly in the weights module — DROP (extractive-span reading comp) and FinanceBench (numeric arithmetic) are structurally blind to the current signal stack. Full writeup: [paper](https://github.com/fathom-lab/styxx/blob/main/papers/cognometry-v0.md).

## Next

- **Manifesto:** https://fathom.darkflobi.com/cognometry — the three laws of cognometry
- **Source:** https://github.com/fathom-lab/styxx — 573 tests, full reproducers
- **Halt policies:** `on_halt='fallback' | 'retry' | 'raise' | 'annotate'`
- **Advanced:** see `docs/` in the repo

If you replicate or disconfirm any of the 8 benchmark numbers, open an issue — we'll cite you in the next paper.